In [25]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.calibration import CalibratedClassifierCV

# =========================
# USTAWIENIA
# =========================

RANDOM_STATE = 42
TARGET_COL = "Is1Winner"

TRAIN_PATH = "total_m.csv"
PRED1_PATH = "predict_m_1.csv"
PRED2_PATH = "predict_m_2.csv"

SUBMISSION_PATH = "submission_m.csv"

# wybrany model: extratrees_cal
best_params_extratrees = {
    "criterion": "log_loss",
    "max_depth": 10,
    "max_features": 0.9,
    "min_samples_leaf": 8,
    "min_samples_split": 8,
    "n_estimators": 421
}

# według wcześniejszego pipeline'u najrozsądniej używać tego:
BEST_FEATURE_SET = "raw_diff"

# kolumny, których nie używamy do modelu
manual_drop_cols = [
    "Season",
    "1TeamID",
    "2TeamID",
    "TeamID_2",
    "ID"
]

In [26]:
# =========================
# FUNKCJE POMOCNICZE
# =========================

def build_feature_set_single(df, target_col, manual_drop_cols, feature_set_name):
    drop_cols = [c for c in manual_drop_cols if c in df.columns]

    if target_col in df.columns:
        drop_cols.append(target_col)

    base_cols = [c for c in df.columns if c not in drop_cols]

    X_raw = df[base_cols].copy()

    cols_2 = [c for c in base_cols if c.endswith("_2")]
    cols_1 = [c[:-2] for c in cols_2 if c[:-2] in base_cols]

    X_diff = pd.DataFrame(index=df.index)
    for c in cols_1:
        X_diff[f"{c}_diff"] = df[c] - df[f"{c}_2"]

    if feature_set_name == "raw":
        return X_raw
    elif feature_set_name == "diff":
        return X_diff
    elif feature_set_name == "raw_diff":
        return pd.concat([X_raw, X_diff], axis=1)
    else:
        raise ValueError(f"Nieznany feature set: {feature_set_name}")


def align_and_clean_features(X_train, X_pred, missing_threshold=0.60, corr_threshold=0.995):
    X_train = X_train.copy()
    X_pred = X_pred.copy()

    # tylko wspólne kolumny
    common_cols = [c for c in X_train.columns if c in X_pred.columns]
    X_train = X_train[common_cols].copy()
    X_pred = X_pred[common_cols].copy()

    # złe kolumny
    all_nan_cols = [c for c in X_train.columns if X_train[c].isna().all()]
    high_missing_cols = [c for c in X_train.columns if X_train[c].isna().mean() > missing_threshold]
    nunique = X_train.nunique(dropna=False)
    constant_cols = nunique[nunique <= 1].index.tolist()

    drop_cols = sorted(set(all_nan_cols + high_missing_cols + constant_cols))

    X_train = X_train.drop(columns=drop_cols, errors="ignore")
    X_pred = X_pred.drop(columns=drop_cols, errors="ignore")

    # imputacja
    imputer = SimpleImputer(strategy="median")

    X_train = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )

    X_pred = pd.DataFrame(
        imputer.transform(X_pred),
        columns=X_pred.columns,
        index=X_pred.index
    )

    # usunięcie bardzo silnie skorelowanych cech - liczone tylko na train
    if X_train.shape[1] > 1:
        corr = X_train.corr(numeric_only=True).abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        high_corr_cols = [c for c in upper.columns if any(upper[c] > corr_threshold)]
    else:
        high_corr_cols = []

    X_train = X_train.drop(columns=high_corr_cols, errors="ignore")
    X_pred = X_pred.drop(columns=high_corr_cols, errors="ignore")

    return X_train, X_pred

In [27]:
# =========================
# WCZYTANIE DANYCH
# =========================

train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
pred1_df = pd.read_csv(PRED1_PATH, low_memory=False)
pred2_df = pd.read_csv(PRED2_PATH, low_memory=False)

pred_df = pd.concat([pred1_df, pred2_df], axis=0, ignore_index=True)

print("train_df:", train_df.shape)
print("pred1_df:", pred1_df.shape)
print("pred2_df:", pred2_df.shape)
print("pred_df :", pred_df.shape)

train_df: (1449, 175)
pred1_df: (33215, 175)
pred2_df: (33215, 175)
pred_df : (66430, 175)


In [28]:
# =========================
# KONTROLA WYMAGANYCH KOLUMN
# =========================

if TARGET_COL not in train_df.columns:
    raise ValueError(f"W total_m.csv brakuje targetu: {TARGET_COL}")

if "ID" not in pred_df.columns:
    raise ValueError("W plikach predict_m_1.csv / predict_m_2.csv brakuje kolumny ID")

In [29]:
# =========================
# BUDOWA CECH
# =========================

y_train = train_df[TARGET_COL].astype(int).copy()

X_train = build_feature_set_single(
    train_df,
    target_col=TARGET_COL,
    manual_drop_cols=manual_drop_cols,
    feature_set_name=BEST_FEATURE_SET
)

X_pred = build_feature_set_single(
    pred_df,
    target_col=TARGET_COL,   # jeśli nie ma tej kolumny, funkcja po prostu jej nie usunie
    manual_drop_cols=manual_drop_cols,
    feature_set_name=BEST_FEATURE_SET
)

X_train, X_pred = align_and_clean_features(X_train, X_pred)

print("X_train:", X_train.shape)
print("X_pred :", X_pred.shape)

X_train: (1449, 249)
X_pred : (66430, 249)


In [30]:
# =========================
# TRENING MODELU
# =========================

base_model = ExtraTreesClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **best_params_extratrees
)

base_model.fit(X_train, y_train)

cal_model = CalibratedClassifierCV(
    estimator=base_model,
    method="sigmoid",
    cv=3
)

cal_model.fit(X_train, y_train)

print("Model wytrenowany.")

Model wytrenowany.


In [31]:
# =========================
# PREDYKCJE
# =========================

pred_proba = cal_model.predict_proba(X_pred)[:, 1]

print("Pred mean:", pred_proba.mean())
print("Pred min :", pred_proba.min())
print("Pred max :", pred_proba.max())

Pred mean: 0.5101981729152689
Pred min : 0.09495574395069623
Pred max : 0.9138223146911422


In [32]:
# =========================
# SUBMISSION: ID, Pred
# =========================

submission = pred_df[["ID"]].copy()
submission["Pred"] = pred_proba

# bezpieczeństwo: zakres [0, 1]
submission["Pred"] = submission["Pred"].clip(0, 1)

# opcjonalnie sprawdzenie
print(submission.head())
print(submission.shape)
print(submission.columns.tolist())

               ID      Pred
0  2026_1101_1102  0.513295
1  2026_1101_1103  0.277999
2  2026_1101_1104  0.196739
3  2026_1101_1105  0.414590
4  2026_1101_1106  0.546871
(66430, 2)
['ID', 'Pred']


In [33]:
# =========================
# SUBMISSION: ID, Pred
# =========================

submission = pred_df[["ID"]].copy()
submission["Pred"] = pred_proba

# bezpieczeństwo: zakres [0, 1]
submission["Pred"] = submission["Pred"].clip(0, 1)

# opcjonalnie sprawdzenie
print(submission.head())
print(submission.shape)
print(submission.columns.tolist())

               ID      Pred
0  2026_1101_1102  0.513295
1  2026_1101_1103  0.277999
2  2026_1101_1104  0.196739
3  2026_1101_1105  0.414590
4  2026_1101_1106  0.546871
(66430, 2)
['ID', 'Pred']


In [34]:
# =========================
# ZAPIS DO CSV
# =========================

submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Zapisano plik submission: {SUBMISSION_PATH}")

Zapisano plik submission: submission_m.csv


In [35]:
import pandas as pd

sub_m = pd.read_csv("submission_m.csv")
sub_w = pd.read_csv("submission_w.csv")

submission_stefan = pd.concat([sub_m, sub_w], axis=0, ignore_index=True)

# dla bezpieczeństwa
submission_stefan = submission_stefan[["ID", "Pred"]]
submission_stefan["Pred"] = submission_stefan["Pred"].clip(0, 1)

submission_stefan.to_csv("submissionStefan.csv", index=False)

print("Zapisano: submissionStefan.csv")
print(submission_stefan.shape)
print(submission_stefan.head())

Zapisano: submissionStefan.csv
(132133, 2)
               ID      Pred
0  2026_1101_1102  0.513295
1  2026_1101_1103  0.277999
2  2026_1101_1104  0.196739
3  2026_1101_1105  0.414590
4  2026_1101_1106  0.546871


In [36]:
print(submission_stefan.columns.tolist())
print(submission_stefan["ID"].isna().sum(), submission_stefan["Pred"].isna().sum())
print(submission_stefan["ID"].duplicated().sum())
print(submission_stefan["Pred"].min(), submission_stefan["Pred"].max())

['ID', 'Pred']
0 0
0
0.0502874119732105 0.9414789701227044


In [37]:
import pandas as pd

pred_w1 = pd.read_csv("predict_m_1.csv")
pred_w2 = pd.read_csv("predict_m_2.csv")

pred_w = pd.concat([pred_w1, pred_w2], axis=0, ignore_index=True)

print("Liczba wierszy:", len(pred_w))
print("Liczba unikalnych ID:", pred_w["ID"].nunique())
print("Duplikaty ID:", pred_w["ID"].duplicated().sum())

print("\nPierwsze 10 ID:")
print(pred_w["ID"].head(10).tolist())

print("\nOstatnie 10 ID:")
print(pred_w["ID"].tail(10).tolist())

print("\nMinimalne ID (leksykograficznie):", pred_w["ID"].min())
print("Maksymalne ID (leksykograficznie):", pred_w["ID"].max())

Liczba wierszy: 66430
Liczba unikalnych ID: 66430
Duplikaty ID: 0

Pierwsze 10 ID:
['2026_1101_1102', '2026_1101_1103', '2026_1101_1104', '2026_1101_1105', '2026_1101_1106', '2026_1101_1107', '2026_1101_1108', '2026_1101_1110', '2026_1101_1111', '2026_1101_1112']

Ostatnie 10 ID:
['2026_1477_1478', '2026_1477_1479', '2026_1477_1480', '2026_1477_1481', '2026_1478_1479', '2026_1478_1480', '2026_1478_1481', '2026_1479_1480', '2026_1479_1481', '2026_1480_1481']

Minimalne ID (leksykograficznie): 2026_1101_1102
Maksymalne ID (leksykograficznie): 2026_1480_1481


In [38]:
import pandas as pd

pred_w1 = pd.read_csv("predict_w_1.csv")
pred_w2 = pd.read_csv("predict_w_2.csv")

pred_w = pd.concat([pred_w1, pred_w2], axis=0, ignore_index=True)

print("Liczba wierszy:", len(pred_w))
print("Liczba unikalnych ID:", pred_w["ID"].nunique())
print("Duplikaty ID:", pred_w["ID"].duplicated().sum())

print("\nPierwsze 10 ID:")
print(pred_w["ID"].head(10).tolist())

print("\nOstatnie 10 ID:")
print(pred_w["ID"].tail(10).tolist())

print("\nMinimalne ID (leksykograficznie):", pred_w["ID"].min())
print("Maksymalne ID (leksykograficznie):", pred_w["ID"].max())

Liczba wierszy: 65703
Liczba unikalnych ID: 65703
Duplikaty ID: 0

Pierwsze 10 ID:
['2026_3101_3102', '2026_3101_3103', '2026_3101_3104', '2026_3101_3105', '2026_3101_3106', '2026_3101_3107', '2026_3101_3108', '2026_3101_3110', '2026_3101_3111', '2026_3101_3112']

Ostatnie 10 ID:
['2026_3477_3478', '2026_3477_3479', '2026_3477_3480', '2026_3477_3481', '2026_3478_3479', '2026_3478_3480', '2026_3478_3481', '2026_3479_3480', '2026_3479_3481', '2026_3480_3481']

Minimalne ID (leksykograficznie): 2026_3101_3102
Maksymalne ID (leksykograficznie): 2026_3480_3481


In [39]:
import pandas as pd

sub_m = pd.read_csv("submission_m.csv")
sub_w = pd.read_csv("submission_w.csv")

print("submission_m shape:", sub_m.shape)
print("submission_w shape:", sub_w.shape)

print("\nDuplikaty w submission_m:", sub_m["ID"].duplicated().sum())
print("Duplikaty w submission_w:", sub_w["ID"].duplicated().sum())

overlap_ids = set(sub_m["ID"]).intersection(set(sub_w["ID"]))
print("Wspólne ID między m i w:", len(overlap_ids))

if len(overlap_ids) > 0:
    print("\nPrzykładowe wspólne ID:")
    print(list(sorted(overlap_ids))[:20])

submission_m shape: (66430, 2)
submission_w shape: (65703, 2)

Duplikaty w submission_m: 0
Duplikaty w submission_w: 0
Wspólne ID między m i w: 0
